In [1]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import MarkdownNodeParser

# 1. LOAD: Read the Markdown files you downloaded from LlamaIndex Cloud
# This looks into your 'data' folder and pulls in all .md files.
reader = SimpleDirectoryReader(input_dir="./data/markdown")
documents = reader.load_data()

# 2. DIVIDE: Use the Markdown headers to split the text logically
# Instead of cutting by character count, it cuts when it sees a new ## or ###
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# 3. VERIFY: Let's see what the "Sections" actually look like
print(f"--- Division Complete ---")
print(f"Total logical sections found: {len(nodes)}\n")

# Peek at the first 3 nodes to see the 'Context' (the headers)
for i, node in enumerate(nodes[:3]):
    # Get the header title from metadata
    header = node.metadata.get("header_path", "Main Title")
    print(f"SECTION {i}: {header}")
    print(f"TEXT PREVIEW: {node.text[:100]}...")
    print("-" * 30)

--- Division Complete ---
Total logical sections found: 165

SECTION 0: /
TEXT PREVIEW: An illustration shows a person lying in bed, covering their face with their hands, with a scribbled ...
------------------------------
SECTION 1: /
TEXT PREVIEW: ## Is it stress or anxiety?

<table>
  <tbody>
    <tr>
        <td>Stress</td>
        <td>Bo...
------------------------------
SECTION 2: /
TEXT PREVIEW: ## Ways to Cope
* Keep a journal.
* Download an app with relaxation exercises.
* Exercise and eat...
------------------------------


* Now that we have our Nodes (sections like "Symptoms" or "Diagnosis"), we face a classic RAG problem: Context Loss. If the AI finds a node that says "Treatment involves CBT and SSRIs," but it doesn't know that node came from the Social Anxiety brochure, it might accidentally use that answer for PTSD.

* Contextual Enrichment solves this by using an LLM to "stamp" each node with its identity. We are basically "gluing" the metadata directly into the text so that when we build our Knowledge Graph in the next phase, the relationships (Triplets) are 100% accurate.

In [12]:

# Verify Ollama is reachable before proceeding
import httpx

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3.5:0.8b"

try:
    resp = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    available = [m["name"] for m in resp.json().get("models", [])]
    if any(MODEL_NAME in m for m in available):
        print(f"✅ Ollama is running and '{MODEL_NAME}' is available.")
    else:
        print(f"⚠️  Ollama is running but '{MODEL_NAME}' was not found.")
        print(f"   Available models: {available}")
        print(f"   Run: ollama pull {MODEL_NAME}")
except Exception as e:
    print(f"❌ Could not reach Ollama at {OLLAMA_BASE_URL}: {e}")
    print("   Make sure Ollama is running: https://ollama.com")


✅ Ollama is running and 'qwen3.5:0.8b' is available.


* we're gonna use different LLMs in experimental results and we're gonna benchmark and compare the results based on accuracy..

In [13]:

# Install the Ollama LlamaIndex integration if not already present
%pip install -q llama-index-llms-ollama

from llama_index.llms.ollama import Ollama

# Initialize the local Ollama LLM
# Increase request_timeout for slower hardware; Ollama defaults to localhost:11434
llm = Ollama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    request_timeout=180.0,   # seconds — raise if your machine is slow
)

# Quick sanity check
test = llm.complete("Reply with one word: ready").text.strip()
print(f"✅ LLM initialized — model response: '{test}'")



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ LLM initialized — model response: 'ready'


In [14]:

# 2. Define the Enrichment Function (Contextual Enrichment)
def enrich_nodes_with_context(nodes):
    enriched_nodes = []
    
    print(f"Starting enrichment for {len(nodes)} nodes...")
    
    for node in nodes:
        # Get the filename or title from metadata to help the LLM
        source_doc = node.metadata.get("file_name", "Unknown Document")
        
        # PROMPT: We ask the LLM to situate this specific chunk
        prompt = f"""
        Below is a section from a mental health brochure titled '{source_doc}'. 
        Please provide a 1-sentence context that describes what disorder and what sub-topic 
        (e.g., Symptoms, Treatment, Causes) this text belongs to. 
        
        TEXT: {node.text[:500]}...
        
        Output only the context sentence and nothing else.
        """
        
        # Generate the tag
        context_tag = llm.complete(prompt).text.strip()
        
        # PREPEND: We add the context to the actual text of the node
        # This is the "magic" that makes GraphRAG much safer.
        node.text = f"[CONTEXT: {context_tag}]\n\n" + node.text
        
        enriched_nodes.append(node)
        print(f"Enriched: {context_tag[:50]}...")
        
    return enriched_nodes

# 3. EXECUTE
enriched_nodes = enrich_nodes_with_context(nodes)


Starting enrichment for 165 nodes...
Enriched: The illustration depicts a mental health condition...


ReadTimeout: timed out

* in the response we want to display the source of the information (source citation)

## Save Cell

In [8]:
import pickle
import os

print("💾 Creating Checkpoints...")
# Create a folder to hold your saved states
os.makedirs("./checkpoints", exist_ok=True)

# 1. Save your Enriched Nodes (So you never have to run enrichment again)
with open("./checkpoints/enriched_nodes_gemini.pkl", "wb") as f:
    pickle.dump(enriched_nodes, f)
    print("✅ Enriched Nodes saved to disk!")

💾 Creating Checkpoints...


NameError: name 'enriched_nodes' is not defined

## Load Cell

In [9]:
import pickle
import os

print("🔄 Loading Checkpoints from disk...")

# 1. Load the Enriched Nodes
if os.path.exists("./checkpoints/enriched_nodes_gemini.pkl"):
    with open("./checkpoints/enriched_nodes_gemini.pkl", "rb") as f:
        enriched_nodes = pickle.load(f)
    print(f"✅ Loaded {len(enriched_nodes)} enriched nodes!")
else:
    print("⚠️ No enriched nodes checkpoint found.")

🔄 Loading Checkpoints from disk...


EOFError: Ran out of input

# Graph construction + Triplets extraction

In [10]:
import re
from llama_index.core.llms import ChatMessage
from llama_index.core import PropertyGraphIndex
from llama_index.core.graph_stores.types import EntityNode, Relation

print("--- INITIATING UPGRADED CUSTOM GRAPH BUILDER ---")

# 1. THE UPGRADED PROMPT (Now asking for Entity Types!)
rigid_prompt = """
You are a medical knowledge graph extractor.
Extract relationships from the text in this EXACT strict format:
(SUBJECT : TYPE) -[RELATION]-> (OBJECT : TYPE)

Allowed Types: CONDITION, SYMPTOM, TREATMENT, MEDICATION, SAFETY_RESOURCE, DEMOGRAPHIC
Allowed Relations: HAS_SYMPTOM, TREATED_BY, PRESCRIBES, SUITABLE_FOR, CONTRAINDICATED_FOR, URGENT_ACTION

Do not explain. Do not add extra text. Output only valid triplets.
If no triplets exist, output nothing.
"""

custom_entities = {}
custom_relations = []

# 2. PROCESS THE CHUNKS
for i, node in enumerate(enriched_nodes[:]):
    print(f"Processing chunk {i+1}...")
    
    system_msg = ChatMessage(role="system", content=rigid_prompt)
    user_msg = ChatMessage(role="user", content=f"Text to extract from:\n{node.text}")
    
    # Talk to Gemini directly
    response = llm.chat([system_msg, user_msg]).message.content
    
    if not response or response.strip() == "None":
        continue
        
    # 3. UPGRADED REGEX: Captures (Subject : Type) -[Relation]-> (Object : Type)
    pattern = r'\(([^:]+)\s*:\s*([^)]+)\)\s*-\[([^]]+)\]->\s*\(([^:]+)\s*:\s*([^)]+)\)'
    matches = re.findall(pattern, response)
    
    for subj, subj_type, rel, obj, obj_type in matches:
        # Clean up strings
        subj = subj.strip().upper()
        subj_type = subj_type.strip().upper()
        rel = rel.strip().upper()
        obj = obj.strip().upper()
        obj_type = obj_type.strip().upper()
        
        # 4. Create proper LlamaIndex Nodes with their Medical Labels!
        subj_node = EntityNode(name=subj, label=subj_type)
        obj_node = EntityNode(name=obj, label=obj_type)
        relation = Relation(source_id=subj_node.id, target_id=obj_node.id, label=rel)
        
        custom_entities[subj_node.id] = subj_node
        custom_entities[obj_node.id] = obj_node
        custom_relations.append(relation)

print(f"\n✅ Extraction finished! Found {len(custom_relations)} raw relations.")

print("5. Indexing data into the Knowledge Graph Database...")
# Create a completely blank index
index = PropertyGraphIndex(nodes=[])

# Force our nodes and relations into the engine
for node_id, entity in custom_entities.items():
    index.property_graph_store.upsert_nodes([entity])
    
index.property_graph_store.upsert_relations(custom_relations)

print("\n✅ Final Database Check:")
all_triplets = index.property_graph_store.get_triplets()
print(f"Total Triplets Successfully Indexed: {len(all_triplets)}\n")

for i, t in enumerate(all_triplets[:15]):
    # Now printing with their actual medical categories!
    print(f" -> ({t[0].name} [{t[0].label}]) -[{t[1].label}]-> ({t[2].name} [{t[2].label}])")

--- INITIATING UPGRADED CUSTOM GRAPH BUILDER ---


NameError: name 'enriched_nodes' is not defined

## Save relations and entities cell

In [ ]:
import pickle
import os

print("💾 Creating Checkpoints...")
# Create a folder to hold your saved states
os.makedirs("./checkpoints", exist_ok=True)

# 1. Save your entities
with open("./checkpoints/custom_entities_gemini.pkl", "wb") as f:
    pickle.dump(custom_entities, f)
    print("✅ Entities saved to disk!")
    
print("💾 Creating Checkpoints...")
# Create a folder to hold your saved states
os.makedirs("./checkpoints", exist_ok=True)

# 2. Save your relations
with open("./checkpoints/custom_relations_gemini.pkl", "wb") as f:
    pickle.dump(custom_relations, f)
    print("✅ Relations saved to disk!")

## Load cell

In [ ]:
import pickle
import os

print("🔄 Loading Checkpoints from disk...")

# 1. Load the Entities
if os.path.exists("./checkpoints/custom_entities_gemini.pkl"):
    with open("./checkpoints/custom_entities_gemini.pkl", "rb") as f:
        custom_entities = pickle.load(f)
    print(f"✅ Loaded {len(custom_entities)} entities!")
else:
    print("⚠️ No entities checkpoint found.")
    
# 2. Load the relations
if os.path.exists("./checkpoints/custom_relations_gemini.pkl"):
    with open("./checkpoints/custom_relations_gemini.pkl", "rb") as f:
        custom_relations = pickle.load(f)
    print(f"✅ Loaded {len(custom_relations)} relations!")
else:
    print("⚠️ No relations checkpoint found.")

In [ ]:
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core import PropertyGraphIndex

print("--- THE CLINICAL FACTS WE EXTRACTED ---")
# Let's print out the first facts directly from memory!
for i, rel in enumerate(custom_relations[:]):
    subj_name = custom_entities[rel.source_id].name
    subj_type = custom_entities[rel.source_id].label
    obj_name = custom_entities[rel.target_id].name
    obj_type = custom_entities[rel.target_id].label
    
    print(f"Fact {i+1}: ({subj_name} [{subj_type}]) -[{rel.label}]-> ({obj_name} [{obj_type}])")

print("\n--- BUILDING THE SEARCH ENGINE ---")
# 1. Initialize the raw database properly
graph_store = SimplePropertyGraphStore()

# 2. Insert our facts into the database
graph_store.upsert_nodes(list(custom_entities.values()))
graph_store.upsert_relations(custom_relations)

# 3. Tell LlamaIndex to build its search engine AROUND our populated database
# (Instead of creating a blank one)
index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm
)

print("✅ Index successfully wrapped around the database!")

# 4. LET'S TEST YOUR RAG!
print("\n🤖 ASKING THE GRAPHRAG A MEDICAL QUESTION:")
query_engine = index.as_query_engine()

question = "Based on the text, what are the symptoms of depression?"
print(f"User: {question}")

# The RAG will now search your triplets to answer!
response = query_engine.query(question)
print(f"GraphRAG: {response}")

# Community Detection (Leiden Algorithm)

This is the **core** of the Microsoft GraphRAG workflow. We:
1. Build a `networkx` graph from our entities and relations
2. Run the **Leiden algorithm** (via `graspologic`) to partition the graph into communities — clusters of tightly-connected entities
3. Each community represents a coherent medical topic (e.g., "Depression treatments", "Anxiety symptoms")

These communities enable **global search**: instead of matching individual triplets, we can reason across entire topic clusters.

In [ ]:
%pip install -q "numpy<2" "graspologic>=3.4.1" "networkx>=3.0"

import importlib
import sys
from collections import defaultdict

import numpy as np
import networkx as nx

# Compatibility shim for packages that try `import numpy.char`
try:
    sys.modules.setdefault("numpy.char", importlib.import_module("numpy.core.defchararray"))
except Exception:
    sys.modules.setdefault("numpy.char", np.char)

from graspologic.partition import leiden

# ============================================================
# COMMUNITY DETECTION — Leiden Algorithm
# ============================================================

# --- 1. Build a NetworkX graph from entities & relations ---
G = nx.Graph()

for eid, entity in custom_entities.items():
    G.add_node(eid, name=entity.name, label=entity.label)

for rel in custom_relations:
    G.add_edge(rel.source_id, rel.target_id, relation=rel.label)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# --- 2. Run Leiden community detection ---
partition = leiden(G, random_seed=42)

# --- 3. Organize communities ---
communities = defaultdict(list)
for node_id, community_id in partition.items():
    communities[community_id].append(node_id)

print(f"\nLeiden found {len(communities)} communities:\n")

# --- 4. Display each community ---
for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):
    member_names = [custom_entities[mid].name for mid in member_ids if mid in custom_entities]
    member_labels = [custom_entities[mid].label for mid in member_ids if mid in custom_entities]
    
    print(f"Community {comm_id} ({len(member_names)} entities):")
    for name, label in zip(member_names, member_labels):
        print(f"  - {name} [{label}]")
    
    # Find internal relations for this community
    community_set = set(member_ids)
    internal_rels = [r for r in custom_relations
                     if r.source_id in community_set and r.target_id in community_set]
    print(f"  Internal relations: {len(internal_rels)}")
    print()

In [ ]:
import networkx as nx
import random

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ============================================================
# COMMUNITY VISUALIZATION WITH MATPLOTLIB + NETWORKX
# ============================================================

# Build the NetworkX graph
G_vis = nx.Graph()

for eid, entity in custom_entities.items():
    G_vis.add_node(eid, name=entity.name, label=entity.label)

for rel in custom_relations:
    if rel.source_id in custom_entities and rel.target_id in custom_entities:
        G_vis.add_edge(rel.source_id, rel.target_id, relation=rel.label)

# Generate distinct colors for each community
random.seed(42)
comm_ids_sorted = sorted(communities.keys(), key=lambda c: -len(communities[c]))
cmap = plt.cm.get_cmap("tab20", max(len(communities), 20))
community_colors_mpl = {}
for idx, comm_id in enumerate(comm_ids_sorted):
    community_colors_mpl[comm_id] = cmap(idx % 20)

# Map node -> community
node_to_comm = {}
for comm_id, member_ids in communities.items():
    for mid in member_ids:
        node_to_comm[mid] = comm_id

# Assign colors & sizes per node
node_colors = []
node_sizes = []
for n in G_vis.nodes():
    comm_id = node_to_comm.get(n, -1)
    node_colors.append(community_colors_mpl.get(comm_id, (0.5, 0.5, 0.5, 1.0)))
    # Size proportional to degree
    node_sizes.append(max(50, G_vis.degree(n) * 15))

# Entity type -> marker (matplotlib doesn't support per-node markers in draw,
# so we'll draw each type separately)
MARKER_MAP = {
    "CONDITION": "D",    # diamond
    "SYMPTOM": "o",      # circle
    "TREATMENT": "s",    # square
    "MEDICATION": "^",   # triangle
    "SAFETY_RESOURCE": "*",  # star
    "DEMOGRAPHIC": "p",  # pentagon
}

# Layout
print("Computing layout (this may take a moment)...")
pos = nx.spring_layout(G_vis, k=0.5, iterations=50, seed=42)

# --- Plot ---
fig, ax = plt.subplots(figsize=(20, 14))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

# Draw edges
nx.draw_networkx_edges(G_vis, pos, ax=ax, alpha=0.15, edge_color="#555555", width=0.5)

# Draw nodes grouped by entity type (for different markers)
for entity_type, marker in MARKER_MAP.items():
    type_nodes = [n for n in G_vis.nodes() if custom_entities.get(n, EntityNode(name="", label="")).label == entity_type]
    if not type_nodes:
        continue
    type_colors = [community_colors_mpl.get(node_to_comm.get(n, -1), (0.5, 0.5, 0.5, 1.0)) for n in type_nodes]
    type_sizes = [max(50, G_vis.degree(n) * 15) for n in type_nodes]
    type_pos = {n: pos[n] for n in type_nodes if n in pos}
    
    xs = [type_pos[n][0] for n in type_nodes if n in type_pos]
    ys = [type_pos[n][1] for n in type_nodes if n in type_pos]
    colors_filtered = [c for n, c in zip(type_nodes, type_colors) if n in type_pos]
    sizes_filtered = [s for n, s in zip(type_nodes, type_sizes) if n in type_pos]
    
    ax.scatter(xs, ys, c=colors_filtered, s=sizes_filtered, marker=marker,
               edgecolors="white", linewidths=0.3, zorder=3, label=entity_type)

# Label only high-degree nodes to avoid clutter
degree_threshold = sorted([G_vis.degree(n) for n in G_vis.nodes()], reverse=True)
cutoff = degree_threshold[min(25, len(degree_threshold) - 1)] if degree_threshold else 0
labels = {n: custom_entities[n].name for n in G_vis.nodes()
          if n in custom_entities and G_vis.degree(n) >= cutoff}
nx.draw_networkx_labels(G_vis, pos, labels, font_size=6, font_color="white",
                        font_weight="bold", ax=ax)

# --- Legend: Entity types ---
type_handles = [plt.Line2D([0], [0], marker=m, color='w', markerfacecolor='gray',
                           markersize=10, linestyle='None', label=t)
                for t, m in MARKER_MAP.items()]

# --- Legend: Top communities ---
top_n_legend = min(10, len(comm_ids_sorted))
comm_handles = [mpatches.Patch(color=community_colors_mpl[comm_ids_sorted[i]],
                               label=f"Community {comm_ids_sorted[i]} ({len(communities[comm_ids_sorted[i]])} nodes)")
                for i in range(top_n_legend)]

legend1 = ax.legend(handles=type_handles, loc="upper left", title="Entity Types",
                    fontsize=8, title_fontsize=9, facecolor="#2a2a4a", edgecolor="white",
                    labelcolor="white")
legend1.get_title().set_color("white")
ax.add_artist(legend1)

legend2 = ax.legend(handles=comm_handles, loc="lower left", title="Top Communities (Leiden)",
                    fontsize=7, title_fontsize=9, facecolor="#2a2a4a", edgecolor="white",
                    labelcolor="white")
legend2.get_title().set_color("white")

ax.set_title("Mental Health Knowledge Graph — Community Detection (Leiden Algorithm)",
             fontsize=16, color="white", fontweight="bold", pad=20)
ax.axis("off")

plt.tight_layout()
plt.savefig("community_graph_matplotlib_gemini.png", dpi=150, facecolor=fig.get_facecolor(),
            bbox_inches="tight")
plt.show()

print(f"\n✅ Visualization saved to community_graph_matplotlib_gemini.png")
print(f"   Nodes: {G_vis.number_of_nodes()} | Edges: {G_vis.number_of_edges()} | Communities: {len(communities)}")

# Community Summarization

For each community detected by Leiden, we use the LLM to generate a natural-language summary. These summaries capture the **theme** of each cluster (e.g., "This community centers on depression treatments including SSRIs, CBT, and ECT").

This is what enables **global queries** — when the user asks a broad question like *"Compare anxiety and depression treatments"*, the system retrieves relevant community summaries instead of individual triplets.

In [ ]:
from llama_index.core.llms import ChatMessage

# ============================================================
# COMMUNITY SUMMARIZATION
# Each community gets a natural-language summary via LLM
# ============================================================

community_summaries = {}

print(f"Summarizing {len(communities)} communities...\n")

for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):
    # --- Gather all entities in this community ---
    members = []
    for mid in member_ids:
        if mid in custom_entities:
            e = custom_entities[mid]
            members.append(f"{e.name} [{e.label}]")
    
    # --- Gather all internal relations ---
    community_set = set(member_ids)
    internal_rels = []
    for r in custom_relations:
        if r.source_id in community_set and r.target_id in community_set:
            src_name = custom_entities[r.source_id].name
            tgt_name = custom_entities[r.target_id].name
            internal_rels.append(f"({src_name}) -[{r.label}]-> ({tgt_name})")
    
    # --- Build the LLM prompt ---
    entities_str = "\n".join(f"  - {m}" for m in members)
    relations_str = "\n".join(f"  - {r}" for r in internal_rels) if internal_rels else "  (no internal relations)"
    
    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a medical knowledge graph analyst. Given a cluster of entities and their "
            "relationships from mental health brochures, write a concise summary (3-5 sentences) "
            "that describes:\n"
            "1. The main disorder(s) or topic this community is about\n"
            "2. Key symptoms, treatments, or medications mentioned\n"
            "3. How the entities are related to each other\n"
            "Be factual and clinical. Do not add information not present in the data."
        )
    )
    
    user_msg = ChatMessage(
        role="user",
        content=(
            f"Summarize this community of medical entities:\n\n"
            f"ENTITIES:\n{entities_str}\n\n"
            f"RELATIONSHIPS:\n{relations_str}"
        )
    )
    
    response = llm.chat([system_msg, user_msg]).message.content.strip()
    community_summaries[comm_id] = {
        "summary": response,
        "entities": members,
        "relations": internal_rels,
        "size": len(members)
    }
    
    print(f"Community {comm_id} ({len(members)} entities):")
    print(f"  Summary: {response[:150]}...")
    print()

print(f"\n✅ Generated summaries for {len(community_summaries)} communities.")

# Upgraded GraphRAG Query Engine (Local + Global Search)

Now we rebuild the index with deduplicated entities and implement a query function that supports **two modes** following Microsoft's workflow:
- **Local search**: Finds specific triplets relevant to the question (good for "What medications treat PTSD?")
- **Global search**: Retrieves community summaries to answer broad questions (good for "Compare treatment approaches across disorders")

In [ ]:
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core import PropertyGraphIndex
from llama_index.core.llms import ChatMessage

# ============================================================
# REBUILD INDEX WITH EXTRACTED DATA
# ============================================================

print("--- REBUILDING INDEX WITH GRAPH DATA ---")

graph_store = SimplePropertyGraphStore()
graph_store.upsert_nodes(list(custom_entities.values()))
graph_store.upsert_relations(custom_relations)

index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm
)

# The local query engine (triplet-level search)
local_query_engine = index.as_query_engine()

print(f"✅ Index rebuilt with {len(custom_entities)} entities, {len(custom_relations)} relations")

# ============================================================
# GLOBAL QUERY FUNCTION (Community-summary-based)
# ============================================================

def global_query(question, community_summaries, llm, top_k=5):
    """
    Microsoft GraphRAG Global Search:
    1. Rank community summaries by relevance to the question
    2. Feed the top-k summaries as context to the LLM
    3. Generate a comprehensive answer
    """
    
    # --- Step 1: Rank communities by relevance (simple keyword overlap for now) ---
    question_lower = question.lower()
    scored = []
    for comm_id, data in community_summaries.items():
        summary = data["summary"].lower()
        entities_text = " ".join(data["entities"]).lower()
        combined = summary + " " + entities_text
        
        # Simple relevance: count how many question words appear in the community
        q_words = set(question_lower.split())
        score = sum(1 for w in q_words if w in combined)
        scored.append((comm_id, score, data))
    
    # Sort by relevance score, take top_k
    scored.sort(key=lambda x: -x[1])
    top_communities = scored[:top_k]
    
    # --- Step 2: Build context from top community summaries ---
    context_parts = []
    for comm_id, score, data in top_communities:
        context_parts.append(
            f"[Community {comm_id} | {data['size']} entities | Relevance: {score}]\n"
            f"{data['summary']}\n"
            f"Key entities: {', '.join(data['entities'][:10])}"
        )
    
    context = "\n\n---\n\n".join(context_parts)
    
    # --- Step 3: Generate answer ---
    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a mental health information assistant. Answer the user's question "
            "based ONLY on the community summaries provided below. These summaries come "
            "from a knowledge graph built from NIMH mental health brochures.\n\n"
            "Be accurate, cite which community the information comes from, and if the "
            "information is not available in the summaries, say so clearly.\n\n"
            f"COMMUNITY SUMMARIES:\n{context}"
        )
    )
    
    user_msg = ChatMessage(role="user", content=question)
    
    response = llm.chat([system_msg, user_msg]).message.content.strip()
    return response, top_communities


def hybrid_query(question, community_summaries, local_query_engine, llm):
    """
    Combines both local (triplet) and global (community) search results.
    """
    print(f"Question: {question}\n")
    
    # Local search
    print("🔍 Local Search (triplet-level)...")
    local_response = local_query_engine.query(question)
    
    # Global search
    print("🌐 Global Search (community-level)...")
    global_response, top_comms = global_query(question, community_summaries, llm)
    
    # Combine
    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a mental health information assistant. Synthesize the following two "
            "responses into a single, comprehensive answer. The LOCAL response comes from "
            "specific knowledge graph triplets. The GLOBAL response comes from community-level "
            "summaries. Combine them without repetition.\n\n"
            f"LOCAL SEARCH RESULT:\n{str(local_response)}\n\n"
            f"GLOBAL SEARCH RESULT:\n{global_response}"
        )
    )
    
    user_msg = ChatMessage(role="user", content=question)
    final = llm.chat([system_msg, user_msg]).message.content.strip()
    
    return {
        "answer": final,
        "local_result": str(local_response),
        "global_result": global_response,
        "communities_used": [(c[0], c[1]) for c in top_comms]
    }


print("✅ Query functions ready: local_query_engine, global_query(), hybrid_query()")

In [ ]:
# ============================================================
# TEST: Local, Global, and Hybrid Queries
# ============================================================

# --- Test 1: LOCAL query (specific, fact-level) ---
print("=" * 60)
print("TEST 1 — LOCAL SEARCH")
print("=" * 60)
q1 = "What medications are used to treat depression?"
local_result = local_query_engine.query(q1)
print(f"Q: {q1}")
print(f"A: {local_result}\n")

# --- Test 2: GLOBAL query (broad, community-level) ---
print("=" * 60)
print("TEST 2 — GLOBAL SEARCH")
print("=" * 60)
q2 = "Compare the treatment approaches for anxiety and depression."
global_result, used_communities = global_query(q2, community_summaries, llm)
print(f"Q: {q2}")
print(f"A: {global_result}")
print(f"Communities used: {[(c[0], c[1]) for c in used_communities]}\n")

# --- Test 3: HYBRID query (combined) ---
print("=" * 60)
print("TEST 3 — HYBRID SEARCH")
print("=" * 60)
q3 = "What are the symptoms of PTSD and how is it treated?"
result = hybrid_query(q3, community_summaries, local_query_engine, llm)
print(f"\nQ: {q3}")
print(f"A: {result['answer']}")

# Graph Visualization (Community-Colored)

Visualize the deduplicated knowledge graph with communities highlighted in different colors using `pyvis`.

In [ ]:
from pyvis.network import Network
import random

# ============================================================
# GRAPH VISUALIZATION WITH COMMUNITY COLORS
# ============================================================

# Generate distinct colors for each community
random.seed(42)
community_colors = {}
for comm_id in communities.keys():
    community_colors[comm_id] = f"#{random.randint(0, 0xFFFFFF):06x}"

# Map node_id -> community_id for quick lookup
node_to_community = {}
for comm_id, member_ids in communities.items():
    for mid in member_ids:
        node_to_community[mid] = comm_id

# Build pyvis network
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True)
net.barnes_hut(gravity=-5000, central_gravity=0.3, spring_length=150)

# Entity type -> shape mapping
SHAPE_MAP = {
    "CONDITION": "diamond",
    "SYMPTOM": "dot",
    "TREATMENT": "square",
    "MEDICATION": "triangle",
    "SAFETY_RESOURCE": "star",
    "DEMOGRAPHIC": "box",
}

# Add nodes
for eid, entity in custom_entities.items():
    comm_id = node_to_community.get(eid, 0)
    color = community_colors.get(comm_id, "#888888")
    shape = SHAPE_MAP.get(entity.label, "dot")
    
    net.add_node(
        eid, 
        label=entity.name, 
        title=f"{entity.name}\nType: {entity.label}\nCommunity: {comm_id}",
        color=color, 
        shape=shape,
        size=20
    )

# Add edges
for rel in custom_relations:
    net.add_edge(rel.source_id, rel.target_id, title=rel.label, label=rel.label, color="#666666")

# Save and display
output_path = "mental_health_knowledge_graph_gemini.html"
net.show(output_path)
print(f"✅ Graph saved to {output_path}")
print(f"   Nodes: {len(custom_entities)} | Edges: {len(custom_relations)} | Communities: {len(communities)}")
print(f"\nLegend:")
print(f"  Diamond = CONDITION | Circle = SYMPTOM | Square = TREATMENT")
print(f"  Triangle = MEDICATION | Star = SAFETY_RESOURCE | Box = DEMOGRAPHIC")
print(f"  Colors represent different Leiden communities")